# Remaining classifier features with fixed DBSCAN labels

Screen the remaining available Part 1 features separately with logistic regression: drug actions, broad ICD categories, protein targets, raw drug names, matched drug names, molecular descriptors from SMILES, and start year. Keep DBSCAN targets/weights, existing embedding scaler, fitting-weight total 157 per reason, final cutoff 0.5, and highest-probability fallback fixed. No clustering, nonlinear models, or feature combinations.

Predeclared grid: **C=100 and C=1000**. For the five categorical fields, compare minimum training-trial counts **3 and 5**. Other representations: one standardized-year block, one compact molecular-descriptor block, two annotation-missing-only controls, and embeddings-only. Fifteen representations × two C settings = **30 configurations**.

Category frequencies and all new preprocessing use the 1,171 training trials only. Raw/matched drug names are lowercased and whitespace-normalized; target symbols are uppercased; repeated values count once per trial. ICD lists are safely parsed and reduced to three-character category prefixes. Biological missing annotations have explicit flags; missing is not interpreted as absence of a drug or target. Molecular features are eight RDKit descriptors, each aggregated by mean and maximum across valid matched structures, plus valid-structure count, no-valid-structure flag, and an invalid-structure flag. Training medians impute absent descriptors; numerical columns are standardized on training only; flags remain binary.

No Kaggle submission, commit, or tag. Preserve exact matrices, targets/weights, preprocessing, models, prediction sets, frequency profiles, molecule parsing audit, and matched-C controls.

In [1]:
from pathlib import Path
from functools import lru_cache
import ast
import hashlib
import json
import platform
import re
import warnings
import joblib
import numpy as np
import pandas as pd
import scipy
from scipy import sparse
import sklearn
from sklearn.base import clone
from sklearn.exceptions import ConvergenceWarning
from sklearn.metrics import f1_score
from sklearn.preprocessing import MultiLabelBinarizer,StandardScaler
from threadpoolctl import threadpool_limits
from IPython.display import display
from rdkit import Chem,rdBase
from rdkit.Chem import Descriptors,Lipinski,rdMolDescriptors

REASONS=["Safety","Funding","Efficacy","Recruitment","Other"]
EMBEDDINGS=[f"emb_{i}" for i in range(150)]
C_GRID=[100.0,1000.0]
MIN_COUNTS=[3,5]
CATEGORICAL_FIELDS=["drug_actions","icdcodes","drug_targets","drugs","matched_drugs"]
ANNOTATION_FIELDS={"drug_actions","drug_targets","matched_drugs"}
cwd=Path.cwd()
DATA=next(p for p in [cwd,cwd/"part1",cwd/"AML-Midterm"/"part1"]
          if (p/"part1_train.csv").is_file() or (p/"data/part1_train.csv").is_file()).resolve()
INPUT_DIR=DATA if (DATA/"part1_train.csv").is_file() else DATA/"data"
SOURCE=DATA/"experiments/classifier_features"
OUT=DATA/"experiments/remaining_classifier_features"
for folder in ["profiles","datasets","vocabularies","models","validation_predictions","submissions"]:(OUT/folder).mkdir(parents=True,exist_ok=True)
train=pd.read_csv(INPUT_DIR/"part1_train.csv")
valid=pd.read_csv(INPUT_DIR/"part1_val.csv")
truth=valid[REASONS].to_numpy(dtype=int)
original_y=train[REASONS].to_numpy(dtype=int)
known=train[REASONS].isin([0,1]).all(axis=1).to_numpy()
assert len(train)==1171 and len(valid)==78 and known.sum()==157
assert train.ID.is_unique and valid.ID.is_unique and set(train.ID).isdisjoint(valid.ID)
assert np.isin(truth,[0,1]).all()
fixed=pd.read_csv(SOURCE/"datasets/fixed_targets_weights.csv",float_precision="round_trip")
assert fixed.ID.tolist()==train.ID.tolist()
Y=fixed[REASONS].to_numpy(dtype=int);W=fixed[[f"weight_{r}" for r in REASONS]].to_numpy()
np.testing.assert_array_equal(Y[known],original_y[known]);np.testing.assert_array_equal(W[known],np.ones((157,5)))
assert np.array_equal(W>0,Y!=-100)
(OUT/"datasets/fixed_targets_weights.csv").write_bytes((SOURCE/"datasets/fixed_targets_weights.csv").read_bytes())
source_model=joblib.load(SOURCE/"models/embeddings_only.joblib")
embedding_scaler=source_model["embedding_scaler"]
source_settings=json.loads((SOURCE/"settings.json").read_text())
reference_sample=float(source_settings["reference_sample_f1"])
reference_macro=float(source_settings["reference_macro_f1"])
protected_paths=[INPUT_DIR/"part1_train.csv",INPUT_DIR/"part1_val.csv",SOURCE/"models/embeddings_only.joblib",
                 SOURCE/"datasets/fixed_targets_weights.csv",SOURCE/"settings.json"]
additional=DATA/"experiments/additional_methods"
for rel in ["datasets/dbscan_m5_q25.csv","labeling_models/dbscan_m5_q25.joblib","classifier_models/dbscan_m5_q25__logistic_C1000.joblib"]:
    protected_paths.append(additional/rel)
protected_hashes={str(p.relative_to(DATA)):hashlib.sha256(p.read_bytes()).hexdigest() for p in protected_paths}
print("Frozen DBSCAN targets and weights; 30 classifier configurations; RDKit",rdBase.rdkitVersion)


Frozen DBSCAN targets and weights; 30 classifier configurations; RDKit 2026.09.1


In [2]:
def normalize_name(value):return " ".join(str(value).strip().lower().split())


def parse_icd(value):
    def flatten(item):
        if isinstance(item,(list,tuple,set)):
            return [z for child in item for z in flatten(child)]
        if isinstance(item,str) and item.strip().startswith("["):
            return flatten(ast.literal_eval(item))
        return [str(item).strip().upper()]
    if pd.isna(value) or not str(value).strip():return []
    return sorted({re.sub(r"[^A-Z0-9]","",v)[:3] for v in flatten(ast.literal_eval(value)) if v})


def categorical_lists(frame,field):
    def parse(value):
        if pd.isna(value) or not str(value).strip():return []
        if field=="icdcodes":return parse_icd(value)
        raw=ast.literal_eval(value) if field=="drugs" else str(value).split(";")
        if field=="drug_targets":return sorted({str(z).strip().upper() for z in raw if str(z).strip()})
        return sorted({normalize_name(z) for z in raw if str(z).strip()})
    return frame[field].map(parse).tolist()


def missing_annotation(frame,field):
    return np.array([not row for row in categorical_lists(frame,field)],dtype=float)[:,None]

CATEGORIES={field:categorical_lists(train,field) for field in CATEGORICAL_FIELDS}
COUNTS={};PROFILE_ROWS=[]
for field,rows in CATEGORIES.items():
    counts=pd.Series([v for row in rows for v in row]).value_counts().rename_axis("category").rename("training_trials")
    known_counts=pd.Series([v for i in np.flatnonzero(known) for v in rows[i]]).value_counts()
    profile=counts.to_frame();profile["known_label_trials"]=known_counts.reindex(counts.index,fill_value=0).astype(int)
    profile.to_csv(OUT/"profiles"/f"{field}_frequencies.csv")
    COUNTS[field]=counts
    for minimum in MIN_COUNTS:
        PROFILE_ROWS.append({"field":field,"minimum_training_trials":minimum,"retained_categories":int((counts>=minimum).sum()),
                             "training_rows_with_no_annotation":sum(not row for row in rows)})
# These three database fields share the same annotation-missing pattern in the training data.
np.testing.assert_array_equal(missing_annotation(train,"matched_drugs"),missing_annotation(train,"drug_targets"))
np.testing.assert_array_equal(missing_annotation(train,"matched_drugs"),train.drug_smiles.fillna("").str.strip().eq("").to_numpy(dtype=float)[:,None])
pd.DataFrame(PROFILE_ROWS).to_csv(OUT/"profiles/categorical_cutoff_sizes.csv",index=False)
display(pd.DataFrame(PROFILE_ROWS))


,field,minimum_training_trials,retained_categories,training_rows_with_no_annotation
0,drug_actions,3,21,516
1,drug_actions,5,17,516
2,icdcodes,3,266,0
3,icdcodes,5,186,0
4,drug_targets,3,230,514
5,drug_targets,5,169,514
6,drugs,3,124,0
7,drugs,5,82,0
8,matched_drugs,3,70,514
9,matched_drugs,5,50,514


In [3]:
DESCRIPTOR_NAMES=["molecular_weight","logP","polar_surface_area","hydrogen_donors","hydrogen_acceptors","rotatable_bonds","ring_count","fraction_sp3"]
NUMERIC_SMILES_NAMES=[f"{aggregation}_{name}" for aggregation in ["mean","max"] for name in DESCRIPTOR_NAMES]+["valid_structure_count"]
SMILES_NAMES=NUMERIC_SMILES_NAMES+["no_valid_structure","has_invalid_structure"]


@lru_cache(maxsize=None)
def molecule_descriptors(smiles):
    with rdBase.BlockLogs():mol=Chem.MolFromSmiles(smiles)
    if mol is None:return None
    values=np.array([Descriptors.MolWt(mol),Descriptors.MolLogP(mol),rdMolDescriptors.CalcTPSA(mol),
                     Lipinski.NumHDonors(mol),Lipinski.NumHAcceptors(mol),Lipinski.NumRotatableBonds(mol),
                     rdMolDescriptors.CalcNumRings(mol),rdMolDescriptors.CalcFractionCSP3(mol)],dtype=float)
    if not np.isfinite(values).all():return None
    return tuple(values)


def raw_smiles_features(frame):
    features=[];audit=[]
    for trial_id,value in zip(frame.ID,frame.drug_smiles):
        parts=[] if pd.isna(value) else sorted({v.strip() for v in str(value).split(";") if v.strip()})
        valid_values=[];invalid=[]
        for part in parts:
            values=molecule_descriptors(part)
            if values is None:invalid.append(part)
            else:valid_values.append(values)
        if valid_values:
            a=np.asarray(valid_values);summaries=np.r_[a.mean(axis=0),a.max(axis=0)]
        else:summaries=np.full(16,np.nan)
        features.append(np.r_[summaries,len(valid_values),float(not valid_values),float(bool(invalid))])
        audit.append({"ID":trial_id,"listed_distinct_structures":len(parts),"valid_structures":len(valid_values),
                      "invalid_structures":len(invalid),"invalid_smiles":";".join(invalid)})
    return np.asarray(features,dtype=float),pd.DataFrame(audit)

TRAIN_SMILES,train_molecule_audit=raw_smiles_features(train)
train_molecule_audit.to_csv(OUT/"profiles/training_smiles_parse_audit.csv",index=False)
assert TRAIN_SMILES.shape==(1171,19)
assert np.isfinite(TRAIN_SMILES[:,16:]).all()
print("SMILES training trials with usable structures:",int((TRAIN_SMILES[:,17]==0).sum()),
      "trials containing invalid structures:",int(TRAIN_SMILES[:,18].sum()))
print("Parsed valid distinct molecular structures:",molecule_descriptors.cache_info())


SMILES training trials with usable structures: 657 trials containing invalid structures: 0
Parsed valid distinct molecular structures: CacheInfo(hits=759, misses=223, maxsize=None, currsize=223)


In [4]:
def make_block(kind,minimum=None,field=None):
    if kind=="none":return {"kind":"none","feature_names":[]}
    if kind=="categorical":
        classes=sorted(COUNTS[field][COUNTS[field]>=minimum].index.tolist())
        encoder=MultiLabelBinarizer(classes=classes,sparse_output=True);encoder.fit(CATEGORIES[field])
        missing_flag=field in ANNOTATION_FIELDS
        names=[f"{field}__{v}" for v in classes]+([f"{field}_annotation_missing"] if missing_flag else [])
        return {"kind":kind,"field":field,"minimum_training_trials":minimum,"encoder":encoder,
                "missing_flag":missing_flag,"feature_names":names}
    if kind=="missing_only":return {"kind":kind,"field":field,"feature_names":[f"{field}_annotation_missing"]}
    if kind=="year":
        scaler=StandardScaler().fit(train[["start_year"]].to_numpy())
        return {"kind":kind,"scaler":scaler,"feature_names":["standardized_start_year"]}
    if kind=="smiles":
        medians=np.nanmedian(TRAIN_SMILES[:,:17],axis=0)
        assert np.isfinite(medians).all()
        numeric=np.where(np.isnan(TRAIN_SMILES[:,:17]),medians[None,:],TRAIN_SMILES[:,:17])
        scaler=StandardScaler().fit(numeric)
        return {"kind":kind,"medians":medians,"scaler":scaler,"descriptor_names":DESCRIPTOR_NAMES,
                "numeric_feature_names":NUMERIC_SMILES_NAMES,"feature_names":SMILES_NAMES}
    raise ValueError(kind)


def transform_block(block,frame):
    kind=block["kind"]
    if kind=="none":return sparse.csr_matrix((len(frame),0))
    if kind=="missing_only":return sparse.csr_matrix(missing_annotation(frame,block["field"]))
    if kind=="year":return sparse.csr_matrix(block["scaler"].transform(frame[["start_year"]].to_numpy()))
    if kind=="smiles":
        raw,_=raw_smiles_features(frame)
        numeric=np.where(np.isnan(raw[:,:17]),block["medians"][None,:],raw[:,:17])
        return sparse.csr_matrix(np.column_stack([block["scaler"].transform(numeric),raw[:,17:]]))
    rows=categorical_lists(frame,block["field"]);allowed=set(block["encoder"].classes_)
    encoded=block["encoder"].transform([[v for v in row if v in allowed] for row in rows]).astype(float).tocsr()
    if block["missing_flag"]:encoded=sparse.hstack([encoded,sparse.csr_matrix(missing_annotation(frame,block["field"]))],format="csr")
    assert encoded.shape[1]==len(block["feature_names"]) and np.isin(encoded.data,[0,1]).all()
    return encoded


def transform_model(model,frame):
    embedding=model["embedding_scaler"].transform(frame[model["embedding_features"]].to_numpy())
    if model["block"]["kind"]=="none":return embedding
    return sparse.hstack([sparse.csr_matrix(embedding),transform_block(model["block"],frame)],format="csr")


def predict_model(model,frame):
    matrix=transform_model(model,frame)
    return np.column_stack([est.predict_proba(matrix)[:,list(est.classes_).index(1)] for est in model["estimators"]])


def label_predictions(probabilities):
    assert np.isfinite(probabilities).all() and ((probabilities>=0)&(probabilities<=1)).all()
    prediction=(probabilities>=0.5).astype(int);empty=prediction.sum(axis=1)==0
    rows=np.flatnonzero(empty);prediction[rows,probabilities[rows].argmax(axis=1)]=1
    return prediction,empty

reference_probability=predict_model(source_model,valid)
reference_prediction,_=label_predictions(reference_probability)
np.testing.assert_allclose(f1_score(truth,reference_prediction,average="samples",zero_division=0),reference_sample,atol=1e-14)
ROWS=[];BASELINE_SCORES={};MATRIX_RECORDS={}


def run_representation(kind,minimum=None,field=None):
    name="embeddings_only" if kind=="none" else (f"{field}_min{minimum}" if kind=="categorical" else (f"{field}_missing_only" if kind=="missing_only" else kind))
    block=make_block(kind,minimum,field)
    template={**source_model,"kind":"remaining_features_logistic","block":block,"estimators":[],
              "source_targets_file":"datasets/fixed_targets_weights.csv","training_matrix_file":f"datasets/{name}.npz",
              "feature_block_weight":1.0}
    matrix=transform_model(template,train);stored=sparse.csr_matrix(matrix)
    assert np.isfinite(stored.data).all()
    names=EMBEDDINGS+block["feature_names"]
    assert stored.shape==(len(train),len(names)) and len(names)==len(set(names))
    sparse.save_npz(OUT/"datasets"/f"{name}.npz",stored)
    (OUT/"datasets"/f"{name}_columns.json").write_text(json.dumps(names,indent=2)+"\n")
    MATRIX_RECORDS[name]={"shape":list(stored.shape),"kind":kind,"field":field,"minimum_training_trials":minimum}
    if kind=="categorical":
        vocabulary=pd.DataFrame({"category":block["encoder"].classes_})
        vocabulary["training_trials"]=vocabulary.category.map(COUNTS[field])
        vocabulary.to_csv(OUT/"vocabularies"/f"{name}.csv",index=False)
    for C in C_GRID:
        experiment=f"{name}_C{C:g}"
        model={**template,"C":C,"estimators":[]}
        with threadpool_limits(limits=1):
            for j in range(5):
                estimator=clone(source_model["estimators"][j]).set_params(C=C)
                included=W[:,j]>0;normalized=W[included,j]*157/W[included,j].sum()
                with warnings.catch_warnings():
                    warnings.simplefilter("error",ConvergenceWarning)
                    estimator.fit(matrix[included],Y[included,j],sample_weight=normalized)
                model["estimators"].append(estimator)
        probability=predict_model(model,valid)
        prediction,empty=label_predictions(probability)
        sample=f1_score(truth,prediction,average="samples",zero_division=0)
        macro=f1_score(truth,prediction,average="macro",zero_division=0)
        if kind=="none":BASELINE_SCORES[C]=sample
        if kind=="none" and C==1000:np.testing.assert_allclose(probability,reference_probability,rtol=1e-8,atol=1e-10)
        row={"experiment":experiment,"representation":name,"feature_group":field if kind=="categorical" else kind,
             "minimum_training_trials":minimum,"C":C,"additional_features":len(block["feature_names"]),
             "total_features":len(names),"sample_f1":sample,"macro_f1":macro,
             "delta_vs_current_leader":sample-reference_sample,"delta_vs_same_C_embeddings":sample-BASELINE_SCORES[C],
             "fallback_trials":int(empty.sum()),"changed_trials_vs_current":int((prediction!=reference_prediction).any(axis=1).sum()),
             **dict(zip([f"f1_{r}" for r in REASONS],f1_score(truth,prediction,average=None,zero_division=0))),
             "is_current":kind=="none" and C==1000,"kaggle_score":np.nan}
        ROWS.append(row)
        frame=valid[["ID"]+REASONS].rename(columns={r:f"actual_{r}" for r in REASONS}).copy()
        for j,r in enumerate(REASONS):frame[f"probability_{r}"]=probability[:,j];frame[f"predicted_{r}"]=prediction[:,j]
        frame["used_fallback"]=empty
        frame["trial_f1"]=2*(truth*prediction).sum(axis=1)/(truth.sum(axis=1)+prediction.sum(axis=1))
        frame.to_csv(OUT/"validation_predictions"/f"{experiment}.csv",index=False)
        joblib.dump(model,OUT/"models"/f"{experiment}.joblib",compress=3)
        pd.DataFrame(ROWS).to_csv(OUT/"comparison.csv",index=False)
        print(f"{experiment}: +{len(block['feature_names'])} features, sample F1={sample:.6f}, macro F1={macro:.6f}, change={sample-reference_sample:+.6f}",flush=True)

run_representation("none")


embeddings_only_C100: +0 features, sample F1=0.888889, macro F1=0.790092, change=-0.021368


embeddings_only_C1000: +0 features, sample F1=0.910256, macro F1=0.811421, change=+0.000000


In [5]:
for field in CATEGORICAL_FIELDS:
    for minimum in MIN_COUNTS:run_representation("categorical",minimum,field)


drug_actions_min3_C100: +22 features, sample F1=0.888889, macro F1=0.800905, change=-0.021368


drug_actions_min3_C1000: +22 features, sample F1=0.880342, macro F1=0.778485, change=-0.029915


drug_actions_min5_C100: +18 features, sample F1=0.888889, macro F1=0.800905, change=-0.021368


drug_actions_min5_C1000: +18 features, sample F1=0.880342, macro F1=0.778485, change=-0.029915


icdcodes_min3_C100: +266 features, sample F1=0.893162, macro F1=0.797110, change=-0.017094


icdcodes_min3_C1000: +266 features, sample F1=0.891026, macro F1=0.793319, change=-0.019231


... (10 outputs left out)

matched_drugs_min3_C100: +71 features, sample F1=0.888889, macro F1=0.750443, change=-0.021368


matched_drugs_min3_C1000: +71 features, sample F1=0.910256, macro F1=0.811421, change=+0.000000


matched_drugs_min5_C100: +51 features, sample F1=0.888889, macro F1=0.750443, change=-0.021368


matched_drugs_min5_C1000: +51 features, sample F1=0.910256, macro F1=0.811421, change=+0.000000


In [6]:
run_representation("missing_only",field="matched_drugs")
run_representation("missing_only",field="drug_actions")
run_representation("year")
run_representation("smiles")


matched_drugs_missing_only_C100: +1 features, sample F1=0.888889, macro F1=0.750443, change=-0.021368


matched_drugs_missing_only_C1000: +1 features, sample F1=0.905983, macro F1=0.805106, change=-0.004274


drug_actions_missing_only_C100: +1 features, sample F1=0.888889, macro F1=0.750443, change=-0.021368


drug_actions_missing_only_C1000: +1 features, sample F1=0.905983, macro F1=0.805106, change=-0.004274


year_C100: +1 features, sample F1=0.826923, macro F1=0.710959, change=-0.083333


year_C1000: +1 features, sample F1=0.750000, macro F1=0.585077, change=-0.160256


smiles_C100: +19 features, sample F1=0.884615, macro F1=0.783430, change=-0.025641


smiles_C1000: +19 features, sample F1=0.884615, macro F1=0.783190, change=-0.025641


In [7]:
comparison=pd.DataFrame(ROWS);assert len(comparison)==30
ranked=comparison.assign(distance_from_current_C=abs(np.log10(comparison.C/1000))).sort_values(
    ["sample_f1","is_current","additional_features","distance_from_current_C","experiment"],ascending=[False,False,True,True,True],kind="stable")
best=ranked.iloc[0]
best_by_group=ranked.groupby("feature_group",sort=False).head(1)
best_by_group.to_csv(OUT/"best_by_group.csv",index=False)
comparison.query("representation=='embeddings_only'").to_csv(OUT/"embedding_controls.csv",index=False)
display(best_by_group[["feature_group","representation","C","additional_features","sample_f1","macro_f1","delta_vs_current_leader"]].round(6))
selected=joblib.load(OUT/"models"/f"{best.experiment}.joblib")
selected_probability=predict_model(selected,valid)
for row in comparison.itertuples(index=False):
    frame=pd.read_csv(OUT/"validation_predictions"/f"{row.experiment}.csv",float_precision="round_trip")
    assert frame.ID.tolist()==valid.ID.tolist()
    probs=frame[[f"probability_{r}" for r in REASONS]].to_numpy()
    predicted=frame[[f"predicted_{r}" for r in REASONS]].to_numpy(dtype=int)
    np.testing.assert_array_equal(label_predictions(probs)[0],predicted)
    np.testing.assert_allclose(f1_score(truth,predicted,average="samples",zero_division=0),row.sample_f1,atol=1e-14)
    np.testing.assert_allclose(f1_score(truth,predicted,average="macro",zero_division=0),row.macro_f1,atol=1e-14)
    loaded=joblib.load(OUT/"models"/f"{row.experiment}.joblib")
    np.testing.assert_allclose(predict_model(loaded,valid),probs,rtol=1e-12,atol=1e-12)
    saved=sparse.load_npz(OUT/loaded["training_matrix_file"])
    expected=sparse.csr_matrix(transform_model(loaded,train));diff=saved-expected
    assert not diff.nnz or abs(diff.data).max()<1e-12
# Independent selected-model refit from exact persisted features and target weights.
replayed=pd.read_csv(OUT/"datasets/fixed_targets_weights.csv",float_precision="round_trip")
assert replayed.ID.tolist()==train.ID.tolist()
np.testing.assert_array_equal(replayed[REASONS].to_numpy(dtype=int),Y)
np.testing.assert_array_equal(replayed[[f"weight_{r}" for r in REASONS]].to_numpy(),W)
matrix=sparse.load_npz(OUT/selected["training_matrix_file"])
if selected["block"]["kind"]=="none":matrix=matrix.toarray()
replay={**selected,"estimators":[]}
with threadpool_limits(limits=1):
    for j in range(5):
        included=W[:,j]>0;estimator=clone(selected["estimators"][j])
        with warnings.catch_warnings():
            warnings.simplefilter("error",ConvergenceWarning)
            estimator.fit(matrix[included],Y[included,j],sample_weight=W[included,j]*157/W[included,j].sum())
        replay["estimators"].append(estimator)
np.testing.assert_allclose(predict_model(replay,valid),selected_probability,rtol=1e-8,atol=1e-10)
np.testing.assert_array_equal(label_predictions(predict_model(replay,valid))[0],label_predictions(selected_probability)[0])
for rel,h in protected_hashes.items():assert hashlib.sha256((DATA/rel).read_bytes()).hexdigest()==h
valid_smiles,valid_parse_audit=raw_smiles_features(valid)
valid_parse_audit.to_csv(OUT/"profiles/validation_smiles_parse_audit.csv",index=False)
print("All 30 saved models/predictions/matrices, reference control, independent winner refit, and protected hashes verified.")


,feature_group,representation,C,additional_features,sample_f1,macro_f1,delta_vs_current_leader
1,none,embeddings_only,1000.0,0,0.910256,0.811421,0.000000
21,matched_drugs,matched_drugs_min5,1000.0,51,0.910256,0.811421,0.000000
25,missing_only,drug_actions_missing_only,1000.0,1,0.905983,0.805106,-0.004274
13,drug_targets,drug_targets_min5,1000.0,170,0.901709,0.803884,-0.008547
17,drugs,drugs_min5,1000.0,82,0.897436,0.803884,-0.012821
9,icdcodes,icdcodes_min5,1000.0,186,0.895299,0.800687,-0.014957
4,drug_actions,drug_actions_min5,100.0,18,0.888889,0.800905,-0.021368
29,smiles,smiles,1000.0,19,0.884615,0.783190,-0.025641
26,year,year,100.0,1,0.826923,0.710959,-0.083333


All 30 saved models/predictions/matrices, reference control, independent winner refit, and protected hashes verified.


In [8]:
candidate=None
if best.sample_f1>reference_sample+1e-12:
    test=pd.read_csv(INPUT_DIR/"part1_test.csv");sample=pd.read_csv(INPUT_DIR/"part1_sample_submission.csv")
    assert test.ID.is_unique and sample.ID.is_unique and set(test.ID)==set(sample.ID)
    assert set(test.ID).isdisjoint(train.ID) and set(test.ID).isdisjoint(valid.ID)
    prediction,empty=label_predictions(predict_model(selected,test))
    output=pd.DataFrame({"ID":test.ID,"Predicted":[" ".join(r for r,yes in zip(REASONS,row) if yes) for row in prediction]})
    output=output.set_index("ID").loc[sample.ID].reset_index()
    assert output.shape==(312,2) and output.Predicted.str.len().gt(0).all()
    path=OUT/"submissions"/f"part1_dbscan_{best.experiment}_submission.csv"
    output.to_csv(path,index=False);pd.testing.assert_frame_equal(pd.read_csv(path),output)
    candidate={"experiment":best.experiment,"validation_sample_f1":float(best.sample_f1),"validation_macro_f1":float(best.macro_f1),
               "submission_file":str(path.relative_to(OUT)),"submission_sha256":hashlib.sha256(path.read_bytes()).hexdigest(),
               "model_file":f"models/{best.experiment}.joblib","model_sha256":hashlib.sha256((OUT/"models"/f"{best.experiment}.joblib").read_bytes()).hexdigest(),
               "test_rows":312,"test_fallback_trials":int(empty.sum()),"kaggle_score":None,"kaggle_status":"prepared locally; not submitted"}
    (OUT/"submission_manifest.json").write_text(json.dumps(candidate,indent=2)+"\n")
settings={"classifier":"logistic regression","C_grid":C_GRID,"categorical_minimum_counts":MIN_COUNTS,
          "categorical_fields":CATEGORICAL_FIELDS,"configurations":30,"reference_sample_f1":reference_sample,"reference_macro_f1":reference_macro,
          "generated_labels_changed":False,"generated_weights_changed":False,"clustering_changed":False,"embedding_scaler_fit_rows":157,
          "new_preprocessing_fit_rows":1171,"categorical_binary_features":"unscaled 0/1","feature_block_weight":1.0,
          "icd_encoding":"three-character alphanumeric category prefixes from safely parsed nested code lists",
          "smiles_descriptors":DESCRIPTOR_NAMES,"smiles_aggregation":["mean","max"],"smiles_extra_features":["valid_structure_count","no_valid_structure","has_invalid_structure"],
          "numeric_imputation":"training medians for missing descriptors","numeric_scaling":"training-only StandardScaler; binary flags unscaled",
          "numeric_imputation_fit_rows":1171,"validation_used_for_fit":False,"validation_rows":78,
          "cutoff":0.5,"fallback":True,"fitting_weight_total_per_reason":157,"matrices":MATRIX_RECORDS,
          "year_scaler_mean":float(joblib.load(OUT/"models/year_C1000.joblib")["block"]["scaler"].mean_[0]),
          "year_scaler_standard_deviation":float(joblib.load(OUT/"models/year_C1000.joblib")["block"]["scaler"].scale_[0]),
          "best_experiment":best.experiment,"best_sample_f1":float(best.sample_f1),"best_macro_f1":float(best.macro_f1),
          "candidate":candidate,"protected_input_sha256":protected_hashes,
          "dataset_file_sha256":{str(p.relative_to(OUT)):hashlib.sha256(p.read_bytes()).hexdigest() for p in sorted((OUT/"datasets").glob('*')) if p.is_file()},
          "selection":"sample F1; ties prefer current leader, fewer features, then original C",
          "versions":{"python":platform.python_version(),"sklearn":sklearn.__version__,"scipy":scipy.__version__,"numpy":np.__version__,"pandas":pd.__version__,"rdkit":rdBase.rdkitVersion},
          "new_kaggle_scores":None,"reported_best_kaggle":0.87393}
(OUT/"settings.json").write_text(json.dumps(settings,indent=2)+"\n")
def table(frame):
    lines=["| Representation | C | Added features | Sample F1 | Macro F1 | Change vs leader | Change vs same-C embeddings |",
           "|---|---:|---:|---:|---:|---:|---:|"]
    for row in frame.itertuples(index=False):
        lines.append(f"| {row.representation} | {row.C:g} | {row.additional_features} | {row.sample_f1:.6f} | {row.macro_f1:.6f} | {row.delta_vs_current_leader:+.6f} | {row.delta_vs_same_C_embeddings:+.6f} |")
    return "\n".join(lines)
lines=["# Remaining classifier feature screen","",
       "30 configurations: five categorical feature fields at minimum training counts 3/5 and C=100/1000, "
       "plus standardized year, molecular descriptors, two missing-annotation controls, and embeddings-only at both C settings. "
       "Each group is added independently to the same 150 embeddings with fixed DBSCAN targets and raw weights.","",
       f"Reference sample F1 **{reference_sample:.6f}**, macro F1 **{reference_macro:.6f}**. "
       f"Selected **{best.experiment}**: sample F1 **{best.sample_f1:.6f}**, macro F1 **{best.macro_f1:.6f}**.","",
       "## Best by feature group","",table(best_by_group),"",
       "## Preprocessing","",
       "Category counts measure distinct training trials. Drug names/actions are lowercased with normalized whitespace; "
       "protein target symbols are uppercased; ICD codes are safely unpacked from nested lists and reduced to three-character "
       "category prefixes. Categories below the selected training frequency are omitted. Multi-hot columns remain 0/1. "
       "Each matched-drug, target, and action block includes its annotation-missing flag. The matched-drug missing indicator "
       "covers the same training rows as missing targets/SMILES; actions have a slightly different pattern. Both patterns "
       "are tested alone so missingness can be separated from biological content.","",
       "Eight RDKit molecular descriptors (molecular weight, logP, polar surface area, hydrogen donors/acceptors, rotatable "
       "bonds, ring count, fraction sp3 carbons) are aggregated by mean and maximum over valid distinct matched SMILES. "
       "Valid-structure count is added; absent descriptors are imputed with training medians. These 17 numeric columns are "
       "standardized using all training trials; no-valid-structure and invalid-structure flags stay binary. No fingerprints "
       "or chemical text bag-of-words is used in this initial screen. Parsing failures are saved in the molecule audit. "
       "Start year is standardized on all 1,171 training trials; the original embedding scaler remains frozen from 157 known-label trials.","",
       "All new preprocessing uses training rows only. Validation/test never supplies vocabulary, imputation, scaling, or fit labels. "
       "The classifier keeps normalized fitting weight 157 per reason, final cutoff 0.5, highest-probability fallback, and block weight 1.","",
       "## All configurations","",table(comparison),"",
       "## Verification and limits","",
       "Every saved model reloads with matching probabilities; all label sets rescore to the recorded sample and macro F1. "
       "Each model recreates its exact saved sparse matrix. The C=1000 embedding control reproduces the original leader, "
       "and the selected model independently refits from its persisted matrix and targets/weights with matching probabilities. "
       "Protected sources retain their checksums.","",
       "This is a bounded screen of specific encodings on the repeatedly consulted 78-trial validation set. It does not "
       "establish independent test improvement or exhaust all encodings, feature combinations, fingerprints, or regularization settings.","",
       "profiles/ and vocabularies/ preserve training frequency distributions, filters, and molecular parsing audits; "
       "datasets/ saves exact matrices, ordered feature names, and shared trial IDs/targets/weights; models/ and "
       "validation_predictions/ preserve all fits and outputs. comparison.csv, best_by_group.csv, embedding_controls.csv, "
       "and settings.json preserve results, settings, and checksums. RDKit was installed in the workspace virtual environment.","",
       "No Kaggle submission, commit, or tag was made.",""]
lines.append(f"New candidate prepared locally: `{candidate['submission_file']}`. Kaggle score unknown." if candidate else
             "No configuration improves the primary validation metric; retain the current embeddings-only leader.")
(OUT/"SUMMARY.md").write_text("\n".join(lines)+"\n")
print(table(best_by_group))
print("Selected:",best.experiment,"Candidate:",candidate)


| Representation | C | Added features | Sample F1 | Macro F1 | Change vs leader | Change vs same-C embeddings |
|---|---:|---:|---:|---:|---:|---:|
| embeddings_only | 1000 | 0 | 0.910256 | 0.811421 | +0.000000 | +0.000000 |
| matched_drugs_min5 | 1000 | 51 | 0.910256 | 0.811421 | +0.000000 | +0.000000 |
| drug_actions_missing_only | 1000 | 1 | 0.905983 | 0.805106 | -0.004274 | -0.004274 |
| drug_targets_min5 | 1000 | 170 | 0.901709 | 0.803884 | -0.008547 | -0.008547 |
| drugs_min5 | 1000 | 82 | 0.897436 | 0.803884 | -0.012821 | -0.012821 |
| icdcodes_min5 | 1000 | 186 | 0.895299 | 0.800687 | -0.014957 | -0.014957 |
| drug_actions_min5 | 100 | 18 | 0.888889 | 0.800905 | -0.021368 | -0.000000 |
| smiles | 1000 | 19 | 0.884615 | 0.783190 | -0.025641 | -0.025641 |
| year | 100 | 1 | 0.826923 | 0.710959 | -0.083333 | -0.061966 |
Selected: embeddings_only_C1000 Candidate: None
